# Word BoW and TF-IDF vectors compared by cosine similarity

Each text of a pair becomes a word BoW or TF-IDF vector, and the pair is "same author" when the
cosine similarity of its two vectors reaches a threshold. The vectorizer hyperparameters are tuned
with Optuna - every trial fits its vectorizer on the training pairs and selects its threshold on the
validation pairs. The vectorizer and the threshold of the best trial then score the test pairs once.

One run tunes one representation. A restarted run continues where it stopped: the study lives in a
SQLite file and a finished run is not repeated. A study is never continued with other settings than
the ones it was started with.

In [ ]:
SMOKE = False  # True - the first rows of every data set, files of their own
N_TRIALS = 30  # finished trials of the study (complete; nothing is pruned)
SEED = 42  # seed of the sampler, the only random part of the run
RUN_TAG = ""  # label in the file names of a repeated run
REPRESENTATION = "tfidf"  # "tfidf" or "bow"

Standard libraries

In [ ]:
import hashlib
import logging
import re
import tempfile
import time
from functools import partial
from itertools import islice
from pathlib import Path

Models, data, plots and Optuna

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns

# numpy 1.26 has no public function that returns this list
from numpy.core._multiarray_umath import (
    __cpu_dispatch__ as numpy_simd_extensions,
)
from numpy.core._multiarray_umath import (
    __cpu_features__ as cpu_has_feature,
)
from optuna.trial import TrialState
from sklearn.metrics import confusion_matrix

The shared modules of the thesis

In [ ]:
from avcore import (
    classifiers,
    data,
    evaluation,
    features,
    progress,
    runs,
    tracking,
    tuning,
)

# Configuration

The vectorizer search space of each representation, every entry suggested in this order

In [ ]:
VECTORIZER_SEARCH_SPACES = {
    "tfidf": {
        "max_features": ("categorical", 20_000, 50_000, 100_000, None),
        "min_df": ("float", 1e-5, 0.01),
        "lowercase": ("categorical", False, True),
        "sublinear_tf": ("categorical", True, False),
    },
    "bow": {
        "max_features": ("categorical", 20_000, 50_000, 100_000, None),
        "min_df": ("float", 1e-5, 0.01),
        "max_df": ("float", 0.7, 1.0),
        "lowercase": ("categorical", False, True),
        "binary": ("categorical", False, True),
    },
}

The function that builds the features of each representation, and its part of the run name

In [ ]:
FEATURE_BUILDERS = {
    "tfidf": features.build_tfidf_features,
    "bow": features.build_bow_features,
}
RUN_NAME_PARTS = {"tfidf": "word_TF-IDF", "bow": "word_BOW"}

The two combinations of the vectors of a pair, and their names in the result tables

In [ ]:
COMBINE_FUNCTIONS = {
    "absolute_difference": features.compute_absolute_difference_features,
    "element-wise": features.compute_element_wise_features,
}
COMBINATION_TABLE_NAMES = {
    "absolute_difference": "absolute_difference",
    "element-wise": "elementwise_product",
}

A representation outside this list stops the run here

In [ ]:
if REPRESENTATION not in VECTORIZER_SEARCH_SPACES:
    raise ValueError(f"Unknown REPRESENTATION {REPRESENTATION!r}.")

The settings of this run

In [ ]:
VECTORIZER_SEARCH_SPACE = VECTORIZER_SEARCH_SPACES[REPRESENTATION]
BUILD_FEATURES = FEATURE_BUILDERS[REPRESENTATION]
# every lowercase choice of the search space (its first item is the kind), each gets token counts
LOWERCASE_VALUES = VECTORIZER_SEARCH_SPACE["lowercase"][1:]
CLASSIFIER = "cosine_similarity"
TRAINING_SECONDS_ATTR = "training_seconds"
DEVELOPER_INITIALS = "JP"

What the run is, in the words of the result tables

In [ ]:
RUN_IDENTITY = {
    "family": "traditional",
    "model": "none",
    "representation": REPRESENTATION,
    "combination": "none",
    "classifier": CLASSIFIER,
}

The rows of a smoke run

In [ ]:
SMOKE_ROWS = {"train": 120, "validation": 40, "test": 40}

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
CLEANED_PATHS = {
    "train": DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE,
    "validation": DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE,
    "test": DATA_DIR / "processed" / data.TEST_CLEANED_FILE,
}

The run name - it is part of every file name of the run

In [ ]:
run_name_parts = [DEVELOPER_INITIALS, "traditional"]
run_name_parts += [RUN_NAME_PARTS[REPRESENTATION], "embedding", CLASSIFIER]
if SMOKE:
    run_name_parts.append("smoke")
if RUN_TAG:
    run_name_parts.append(str(RUN_TAG))
run_name_parts.append("v2")
RUN_NAME = "_".join(run_name_parts)

A run name that is not a plain file name stops the run

In [ ]:
if not re.fullmatch(r"[\w.-]+", RUN_NAME):
    raise ValueError(
        f"Run name {RUN_NAME!r} is not a plain file name - change RUN_TAG.",
    )

Every file the run writes

In [ ]:
STUDY_PATH = DATA_DIR / "studies" / f"optuna_study_{RUN_NAME}.db"
VECTORIZER_PATH = DATA_DIR / "models" / f"{RUN_NAME}_best_vectorizer.pkl"
# the result tables are built from results_v2, so a smoke run writes into a folder of its own
RESULTS_DIR = Path.home() / "results_v2"
if SMOKE:
    RESULTS_DIR = Path.home() / "results_v2_smoke" / str(RUN_TAG)
RUN_DIR = RESULTS_DIR / RUN_NAME
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

Run context - the commit, the hashes of the lock file and of the data and the versions.
It refuses a repository with uncommitted or unpushed changes.

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
run_context = tracking.collect_run_context(PROJECT_DIR, list(CLEANED_PATHS.values()))
print(f"Run {RUN_NAME}, commit {run_context.environment['commit'][:7]}")
# where the run stands, for watch_progress.py on dt - the run never reads it back
progress_file = progress.ProgressFile(
    Path.home() / "logs" / f"{RUN_NAME}.progress.json",
    RUN_NAME,
)
progress_file.start_stage("token counts")

Numpy uses the fastest instruction sets this CPU has (SIMD extensions such as AVX2 or AVX-512). Different instruction sets give slightly different np.log results and can change which terms max_features keeps. The list is therefore saved with the study, and a study is continued only on a CPU with the same list.

In [ ]:
simd_extensions_on_this_cpu = []
for extension in numpy_simd_extensions:
    if cpu_has_feature[extension]:
        simd_extensions_on_this_cpu.append(extension)
CPU_SIMD_FOUND = " ".join(simd_extensions_on_this_cpu)

# Settings

Everything that is fixed before the first trial. The pair counts are left out, because the data
file hashes below already fix the pairs - so a run with different settings stops here, before the
texts are counted.

In [ ]:
run_settings = {
    **RUN_IDENTITY,
    "smoke": SMOKE,
    "seed": SEED,
    "n_trials": N_TRIALS,
    "mlflow_experiment": EXPERIMENT_NAME,
    "study_name": STUDY_PATH.stem,
    "decision_rule": "cosine similarity >= threshold",
    "threshold_candidates": "every validation similarity",
    "cpu_simd_found": CPU_SIMD_FOUND,
}

Add the searched ranges and, in a smoke run, the number of rows read from each file

In [ ]:
for name, space in VECTORIZER_SEARCH_SPACE.items():
    run_settings[f"search.{name}"] = space
if SMOKE:
    for split, n_rows in SMOKE_ROWS.items():
        run_settings[f"smoke_rows.{split}"] = n_rows

The study also stores the hashes of the data files and of uv.lock, because the token counts depend on both

In [ ]:
study_settings = dict(run_settings)
study_settings["uv_lock_sha256"] = run_context.environment["uv_lock_sha256"]
for path in CLEANED_PATHS.values():
    hash_key = f"data_sha256.{path.name}"
    study_settings[hash_key] = run_context.environment[hash_key]

A study or a result folder with different settings stops the run here, before anything is counted, trained or written.

In [ ]:
runs.check_study_settings(STUDY_PATH, study_settings)
# find_finished_run refuses a result folder of this run with other settings;
# a finished one without its study would run the whole search again
if runs.find_finished_run(RUN_DIR, run_settings) and not STUDY_PATH.is_file():
    raise RuntimeError(
        f"{RUN_DIR} exists but its study {STUDY_PATH.name} does not - move the folder away.",
    )

# Token counts

Counting the words of all texts is the slowest step, so it is done here once instead of in every
trial. The only setting that changes the counting is lowercase, so the texts are counted once for
each lowercase value. Each trial then builds its TF-IDF or BoW features from these counts.

A smoke run counts only copies of the first lines of every file, kept in a temporary folder

In [ ]:
def copy_first_lines(path: Path, n_lines: int, target_dir: Path) -> Path:
    """Copy the first n_lines of a file into target_dir, under the same file name."""
    with path.open(encoding="utf-8") as file:
        first_lines = list(islice(file, n_lines))
    copy_path = target_dir / path.name
    copy_path.write_text("".join(first_lines), encoding="utf-8")
    return copy_path

In [ ]:
temp_dir = tempfile.TemporaryDirectory()
if SMOKE:
    files_to_count = {}
    for split, path in CLEANED_PATHS.items():
        files_to_count[split] = copy_first_lines(
            path,
            SMOKE_ROWS[split],
            Path(temp_dir.name),
        )
else:
    files_to_count = CLEANED_PATHS

Count the words of every text, once for each lowercase value (True or False)

In [ ]:
token_counts = {}
for lowercase in LOWERCASE_VALUES:
    token_counts[lowercase] = features.build_token_counts(
        files_to_count,
        lowercase=lowercase,
    )
temp_dir.cleanup()

The pairs of every split - both lowercase values read the same lines, so the first one is used

In [ ]:
count_table = token_counts[LOWERCASE_VALUES[0]]
pairs = {}
for split in features.SPLITS:
    split_counts = count_table.splits[split]
    pairs_df = split_counts.pairs_df
    pairs[split] = pairs_df
    print(f"{split}: {len(pairs_df)} pairs")

The prediction tables are keyed by pair id, so a repeated validation or test id stops the run

In [ ]:
for split in ["validation", "test"]:
    ids_are_unique = pairs[split]["id"].is_unique
    if not ids_are_unique:
        raise ValueError(f"The {split} pairs have repeated ids.")

# Hyperparameter optimization

Each trial first builds its features - it takes the count table of its lowercase value and turns it
into TF-IDF or BoW features with its other vectorizer hyperparameters

In [ ]:
def build_pair_features(
    token_counts: dict[bool, features.PairMatrices],
    vectorizer_params: dict[str, object],
    splits: tuple[tuple[str, ...], str],
) -> features.PairFeatures:
    """Build the TF-IDF or BoW features of the fit and of the target pairs of one trial.

    Args:
        token_counts: The count table of every lowercase value.
        vectorizer_params: The vectorizer hyperparameters of the trial, lowercase included.
        splits: features.TUNING_SPLITS (fit on train, score validation) or
            features.HELD_OUT_SPLITS (fit on train, score test).

    Returns:
        The features of both texts of the fit and of the target pairs.
    """
    # lowercase picks the count table, the rest goes to the feature builder
    builder_params = dict(vectorizer_params)
    lowercase = builder_params.pop("lowercase")
    fit_splits, target_split = splits
    return BUILD_FEATURES(
        token_counts[lowercase],
        fit_splits,
        target_split,
        **builder_params,
    )

The cosine similarity of the two texts of every target pair, next to its id and label

In [ ]:
def score_target_pairs(pair_features: features.PairFeatures) -> pd.DataFrame:
    """Return the target pairs with their cosine similarity in the column score.

    Args:
        pair_features: Output of build_pair_features.

    Returns:
        One row per target pair with the columns id, same and score.
    """
    similarities = features.compute_cosine_similarities(pair_features.target_sides)
    scores_df = pair_features.target_pairs_df.copy()
    scores_df["score"] = similarities
    return scores_df

The threshold of the highest F1 among every observed similarity (the lowest one if two are equal),
with the F1 of every candidate

In [ ]:
def select_threshold(scores_df: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    """Evaluate every observed score as a threshold and return the best row and all rows.

    Args:
        scores_df: Output of score_target_pairs for the validation pairs.

    Returns:
        The row of the best threshold and the table of every candidate threshold.
    """
    candidates = np.unique(scores_df["score"]).tolist()
    thresholds_df = evaluation.evaluate_classification_thresholds(
        scores_df["same"].astype(int),
        scores_df["score"],
        candidates,
    )
    best_row = evaluation.select_best_threshold(thresholds_df)
    return best_row, thresholds_df

The objective function of one trial - it builds the vectors on the training pairs, scores the
validation pairs and returns the F1 of the best threshold, which Optuna maximizes. The F1 is
returned unrounded, so two trials are never tied only by rounding.

In [ ]:
def objective(
    trial: optuna.Trial,
    *,
    token_counts: dict[bool, features.PairMatrices],
    progress_file: progress.ProgressFile,
) -> float:
    """Fit the vectorizer of one trial and select its threshold on the validation pairs.

    Args:
        trial: Trial started by tuning.run_study.
        token_counts: The count table of every lowercase value.
        progress_file: Shows the running trial and the finished trials.

    Returns:
        The unrounded F1 (a fraction) of the same-author class at the best validation threshold;
        the threshold and the rounded percent metrics are stored as user attributes.
    """
    progress_file.start_stage("tuning", **progress.trial_fields(trial, N_TRIALS))
    vectorizer_params = classifiers.suggest_params(trial, VECTORIZER_SEARCH_SPACE)

    started = time.perf_counter()
    pair_features = build_pair_features(
        token_counts,
        vectorizer_params,
        features.TUNING_SPLITS,
    )
    validation_scores_df = score_target_pairs(pair_features)
    best_row, thresholds_df = select_threshold(validation_scores_df)
    trial.set_user_attr(TRAINING_SECONDS_ATTR, time.perf_counter() - started)

    # max_features is only a cap, so the real vocabulary size is logged
    first_text_features = pair_features.fit_sides[0]
    trial.set_user_attr("n_features", first_text_features.shape[1])
    trial.set_user_attr("threshold", float(best_row["threshold"]))
    trial.set_user_attr("n_threshold_candidates", len(thresholds_df))
    for name in evaluation.METRIC_COLUMNS:
        trial.set_user_attr(f"validation_{name}", float(best_row[name]))
    print(f"Trial {trial.number}: validation F1 {best_row['f1']}")
    return float(best_row["f1_unrounded"])

Open the study file, or continue it if it already holds trials, and store the settings in it on the first run. The trials report no intermediate values, so nothing is pruned (NopPruner).

In [ ]:
study = tuning.open_study(STUDY_PATH, pruner=optuna.pruners.NopPruner(), seed=SEED)
runs.store_study_settings(study, study_settings)

Run trials until the study holds N_TRIALS finished ones. Optuna calls the objective with the trial only, so partial fixes its other two arguments.

In [ ]:
tuning.run_study(
    study,
    partial(objective, token_counts=token_counts, progress_file=progress_file),
    n_trials=N_TRIALS,
)

Stop if the study ends with another number of finished trials, for example a study started with a larger N_TRIALS

In [ ]:
finished_trials = study.get_trials(deepcopy=False, states=tuning.FINISHED_STATES)
if len(finished_trials) != N_TRIALS:
    raise RuntimeError(
        f"The study holds {len(finished_trials)} finished trials, not {N_TRIALS}.",
    )

The best trial (the highest validation F1, the earlier trial on a tie) and the trials that completed, failed and crashed

In [ ]:
best_trial = runs.select_best_trial(study)
complete_trials = study.get_trials(deepcopy=False, states=(TrialState.COMPLETE,))
failed_trials = study.get_trials(deepcopy=False, states=(TrialState.FAIL,))
# a crashed attempt (killed pod, OOM) is repeated with the same params - not a real failure
crashed_trials = []
for trial in failed_trials:
    if trial.user_attrs.get("crashed"):
        crashed_trials.append(trial)
tuning_training_seconds = runs.sum_training_seconds(study, TRAINING_SECONDS_ATTR)

The trials table, with the best trial marked

In [ ]:
trials_df = study.trials_dataframe()
trials_df["is_best_trial"] = trials_df["number"] == best_trial.number
best_f1 = best_trial.user_attrs["validation_f1"]
print(f"Best trial {best_trial.number}: validation F1 {best_f1}, {best_trial.params}")

# The final model

The final model IS the best trial - its vectorizer, fitted on the training texts only, and the
threshold it selected on the validation pairs. Nothing is refitted, so the threshold belongs to
exactly the vectors it is applied to and was chosen on pairs that vectorizer never saw.

In [ ]:
best_vectorizer_params = dict(best_trial.params)
threshold = best_trial.user_attrs["threshold"]

The params of the run - its settings, the pair counts and the hyperparameters of the final model

In [ ]:
run_params = dict(run_settings)
for split, pairs_df in pairs.items():
    run_params[f"{split}_pairs"] = len(pairs_df)
# the vectorizer of the best trial is fitted on the training texts only
run_params["final_train_pairs"] = len(pairs["train"])
run_params["best_trial_number"] = best_trial.number
# the trial number also counts crashed attempts, the position does not
run_params["best_trial_position"] = best_trial.user_attrs["position"]
for name, value in best_trial.params.items():
    run_params[f"final_{name}"] = value

If the result folder of this run already exists with these params, the run finished before - the cells below then do nothing, as every one starts with if not run_is_complete

In [ ]:
run_is_complete = runs.find_finished_run(RUN_DIR, run_params)
if run_is_complete:
    print(f"{RUN_DIR} exists - the final model is already trained and evaluated.")

Score the validation pairs with the best trial again - the tables of the run need these scores, and the same threshold has to come out

In [ ]:
if not run_is_complete:
    progress_file.start_stage("final model")
    # the count table of the other lowercase value is not needed any more (about 5 GB)
    token_counts.pop(not best_vectorizer_params["lowercase"], None)
    validation_features = build_pair_features(
        token_counts,
        best_vectorizer_params,
        features.TUNING_SPLITS,
    )
    validation_scores_df = score_target_pairs(validation_features)
    validation_best_row, thresholds_df = select_threshold(validation_scores_df)
    if float(validation_best_row["threshold"]) != threshold:
        raise RuntimeError(
            f"The best trial selected {threshold}, its rebuild selects {validation_best_row['threshold']}.",
        )
    # the training features are as large as the counts and the test build allocates them again
    del validation_features

Build the vectors of the test pairs with the vectorizer of the best trial - fitted on the training texts, the test texts are only transformed

In [ ]:
if not run_is_complete:
    started = time.perf_counter()
    final_features = build_pair_features(
        token_counts,
        best_vectorizer_params,
        features.HELD_OUT_SPLITS,
    )
    final_training_seconds = time.perf_counter() - started

Save the vectorizer - with the threshold it turns two raw texts into a decision

In [ ]:
if not run_is_complete:
    joblib.dump(final_features.vectorizer, VECTORIZER_PATH)
    # ties the result folder to the vectorizer file the web app loads
    with VECTORIZER_PATH.open("rb") as file:
        run_params["vectorizer_sha256"] = hashlib.file_digest(
            file,
            "sha256",
        ).hexdigest()

# Evaluate on the test pairs

The test pairs are scored once and never select anything.

In [ ]:
if not run_is_complete:
    progress_file.start_stage("test scoring", n_pairs=len(pairs["test"]))
    started = time.perf_counter()
    test_scores_df = score_target_pairs(final_features)
    scoring_seconds_test = time.perf_counter() - started

The prediction tables (one row per pair - id, same, score, pred) and the test metrics

In [ ]:
if not run_is_complete:
    validation_predictions_df = runs.build_predictions(
        pairs["validation"],
        validation_scores_df,
        threshold,
    )
    predictions_df = runs.build_predictions(pairs["test"], test_scores_df, threshold)
    test_metrics = evaluation.evaluate_results(
        predictions_df["same"],
        predictions_df["pred"],
    )
    print(test_metrics)

# Create the confusion matrix

The figure

In [ ]:
if not run_is_complete:
    confusion_figure, axis = plt.subplots(figsize=(6, 5))

The counts of the test pairs

In [ ]:
if not run_is_complete:
    sns.heatmap(
        confusion_matrix(predictions_df["same"], predictions_df["pred"], labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[False, True],
        yticklabels=[False, True],
        ax=axis,
        cbar=False,
    )

The labels

In [ ]:
if not run_is_complete:
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_title(f"Confusion Matrix: {RUN_NAME}")

# Plot the F1 against the threshold

The F1 of every candidate threshold on the validation pairs, the selected threshold dashed

In [ ]:
if not run_is_complete:
    threshold_figure, threshold_axis = plt.subplots(figsize=(9, 5))
    threshold_axis.plot(thresholds_df["threshold"], thresholds_df["f1"], linewidth=2)
    threshold_axis.axvline(threshold, color="grey", linestyle="--")

The labels

In [ ]:
if not run_is_complete:
    threshold_axis.set_xlabel("Threshold")
    threshold_axis.set_ylabel("Validation F1 (%)")
    threshold_axis.set_title(f"F1 vs Threshold: {RUN_NAME}")
    threshold_axis.grid(True)

# Save the results

Everything the run reports is written into one folder; it is uploaded to MLflow from there.

The metrics of the run

In [ ]:
if not run_is_complete:
    run_metrics = {
        "best_trial_validation_f1": best_trial.user_attrs["validation_f1"],
        "n_trials_complete": len(complete_trials),
        "n_trials_pruned": len(finished_trials) - len(complete_trials),
        "n_trials_failed": len(failed_trials) - len(crashed_trials),
        "n_trials_crashed": len(crashed_trials),
        "training_seconds_tuning": tuning_training_seconds,
        "training_seconds_final": final_training_seconds,
        "scoring_seconds_test": round(scoring_seconds_test, 1),
        "final_n_features": len(final_features.vectorizer.vocabulary_),
        # no effect on the results - logged to compare the times of two runs
        "n_jobs": N_JOBS,
    }

The test metrics and the solver iterations of the final model

In [ ]:
if not run_is_complete:
    for name, value in test_metrics.to_dict().items():
        run_metrics[f"test_{name}"] = value
    final_n_iter = count_iterations(final_classifier)
    if final_n_iter is not None:
        run_metrics["final_n_iter"] = final_n_iter

Write the result folder - a half-written .partial folder left by a killed save is removed first

In [ ]:
if not run_is_complete:
    runs.remove_unfinished_run_dir(RUN_DIR)
    tracking.write_run_files(
        RUN_DIR,
        context=run_context,
        params=run_params,
        metrics=run_metrics,
        tables={"trials": trials_df, "predictions-test": predictions_df},
        figures={"confusion_matrix-test": confusion_figure},
    )

The finished run - the result folder and its metrics

In [ ]:
progress_file.start_stage("done")
print(RUN_DIR)
print((RUN_DIR / tracking.METRICS_FILE).read_text(encoding="utf-8"))

In [ ]:
print("hello werld")